In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

In [ ]:
# Set Working Directory

In [ ]:
# Import sales data
restaurant_example = pd.read_excel('palate_data_excel/batch_1/orders_item_level/SAFK7ND1HR6XS.xlsx')

# Typing
restaurant_example['item_price'] = restaurant_example['item_price'].astype(int)
restaurant_example['item_quantity'] = restaurant_example['item_quantity'].astype('float')

# Make time series
restaurant_example['created_at'] = restaurant_example['created_at'].str.replace(r'\s[A-Z]{3}', '', regex=True) # Timezone is saved with a number format, so the alphabetical one is unnecessary
restaurant_example['created_at'] = pd.to_datetime(restaurant_example['created_at'], utc=True)
restaurant_example.set_index(keys="created_at", drop=True, inplace=True)
restaurant_example.sort_index(inplace=True)

In [ ]:
# Clean sales
restaurant_example = restaurant_example[~(restaurant_example['item_name'] == 'nan')] # Remove NA-likes
restaurant_example.dropna(inplace=True) # Remove NA
restaurant_example.drop_duplicates(inplace=True) # Drop perfect duplicates
restaurant_example.loc[:,'item_name'] = restaurant_example.loc[:,'item_name'].str.title() # Standardize the capitalization scheme
restaurant_example.drop_duplicates(subset=['order_id','item_name'], inplace=True) # If there is a row with the same item and order id, then it should be a duplicate

In [ ]:
# Import promotional items data
before_after_details = pd.read_excel('palate_data_excel/batch_1/before_after_details.xlsx')
before_after_details.set_index('location_id', inplace=True)

In [ ]:
# Boundaries for visuals
start_date = pd.Timestamp('2013-02-28 20:25:33')
start_date = start_date.tz_localize(None) # Remove timezone info
end_date = pd.to_datetime("now") # No timezone info because we're converting to weeks

In [ ]:
# Determine weeks active
active_weeks = {}
restaurant_example.index = restaurant_example.index.tz_localize(None) # Remove timezone info
restaurant_example['Week'] = restaurant_example.index.to_period('W')
active_weeks = set(restaurant_example.groupby('Week').size().index.tolist())

In [ ]:
# Visualize
plt.figure(figsize=(14, 8))

for week in active_weeks:
    plt.hlines(y='SAFK7ND1HR6XS', xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label='SAFK7ND1HR6XS')

promo_datetime = pd.to_datetime(before_after_details.loc['SAFK7ND1HR6XS', 'cross_over_date'])
plt.plot(promo_datetime, 'SAFK7ND1HR6XS', 'ro')

plt.title('Weekly Activity for Example Restaurant with Gaps for Inactive Weeks')
plt.xlabel('Date')
plt.ylabel('Restaurant ID')
plt.xlim(start_date, end_date)
plt.tight_layout()
plt.show()